# 01-Load-Data



## Question

How do I load in the NBA data and validate it?

---

Next cell imports python things and defines where the data resides. Note that the data_archive directory is in the .gitignore file so it is only on the locan machine where I am running and not on GitHub.

In [1]:
# Imports (all at the top) and parameters (one place, near the top).
from pathlib import Path
import json
import re

import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt

# import myproject  # reusable logic lives in src/ — import it, don't inline it

DATA_DIR = Path("..")/"data_archive"
DATA_META= DATA_DIR/"nba-play-by-play-data-1997-2023-metadata.json"

The next cell takes a look at the directory of files to see what is here. I just load a few of the columnes into a Pandas Dataframe to show something that might be useful in a table. This is purely exporatory and required me to know that the filename and the season info was going to be there.

In [3]:
csv_files = sorted(DATA_DIR.glob("pbp*.csv"))

inventory = pd.DataFrame({
    "file": [p.name for p in csv_files],
    "path": [str(p) for p in csv_files],
    "size_mb": [p.stat().st_size / 1e6 for p in csv_files],
    "season_from_filename": [
        int(re.search(r"pbp(\d{4})\.csv", p.name).group(1))
        for p in csv_files
    ],
})

inventory_preview = pd.concat([
    inventory.head(5),
    inventory.tail(5)
])

#display(inventory_preview) 
# I used pd.concat to only show the start (head) and end (tail) five lines in the inventory table.
#if you want to see the whole table, uncomment the next line instead of the display('inventory_preview') line.

display(inventory) 

,file,path,size_mb,season_from_filename
0,pbp1997.csv,../data_archive/pbp1997.csv,67.595210,1997
1,pbp1998.csv,../data_archive/pbp1998.csv,69.083950,1998
2,pbp1999.csv,../data_archive/pbp1999.csv,43.125239,1999
3,pbp2000.csv,../data_archive/pbp2000.csv,70.531910,2000
4,pbp2001.csv,../data_archive/pbp2001.csv,69.229124,2001
5,pbp2002.csv,../data_archive/pbp2002.csv,68.650589,2002
6,pbp2003.csv,../data_archive/pbp2003.csv,69.609204,2003
7,pbp2004.csv,../data_archive/pbp2004.csv,69.165167,2004
8,pbp2005.csv,../data_archive/pbp2005.csv,72.163560,2005
9,pbp2006.csv,../data_archive/pbp2006.csv,71.961720,2006


### Clean up

Data collected over a long period of time is rarely consistent.  That means one of the first things you need to do is explore a little to see what you have and then make a plan to clean it up if needed.  This clean up is often called "data wrangling" or just "wrangling."  

This dataset came with a JSON file which is likely metadata so let's take a look and see what it tell us. 

In [4]:
with open(DATA_META, "r") as f:
    meta = json.load(f)

schema_rows = []
for record_set in meta.get("recordSet", []):
    file_name = record_set.get("name")
    for field in record_set.get("field", []):
        schema_rows.append({
            "file": file_name,
            "column": field.get("name"),
            "data_type": ", ".join(field.get("dataType", [])),
            "description": field.get("description", ""),
        })

schema_df = pd.DataFrame(schema_rows)
display(schema_df.head(30))
display(schema_df.groupby("column")["data_type"].agg(lambda x: sorted(set(x))))

,file,column,data_type,description
0,pbp1997.csv,gameid,sc:Text,NBA.com Game ID
1,pbp1997.csv,period,sc:Integer,Quarter
2,pbp1997.csv,clock,sc:Text,
3,pbp1997.csv,h_pts,sc:Float,Current Home Points
4,pbp1997.csv,a_pts,sc:Float,Current Away Points
5,pbp1997.csv,team,sc:Text,
6,pbp1997.csv,playerid,sc:Text,NBA.com Player ID
7,pbp1997.csv,player,sc:Text,Player's Name
8,pbp1997.csv,type,sc:Text,Type of Event
9,pbp1997.csv,subtype,sc:Text,Subtype of Event


column
a_pts       [sc:Float, sc:Integer]
clock                    [sc:Text]
desc                     [sc:Text]
dist        [sc:Float, sc:Integer]
gameid                   [sc:Text]
h_pts       [sc:Float, sc:Integer]
period                [sc:Integer]
player                   [sc:Text]
playerid                 [sc:Text]
result                   [sc:Text]
season                [sc:Integer]
subtype                  [sc:Text]
team                     [sc:Text]
type                     [sc:Text]
x           [sc:Float, sc:Integer]
y           [sc:Float, sc:Integer]
Name: data_type, dtype: object

Right away we see schema drift. (In data work, "schema" means the organization of the data and the names used to describe the fields. It acts as the blueprint of a data structure whether that is a database or just a table.) In this dataset, the JSON metadata, for example, shows that coordinate fields appear as integers in many seasons but floats in some newer seasons; that will cause trouble if you try to do math on them so this exploration tells you quickly that you are going to have to clean that up.  You could, for example,standardize x, y, and dist to "nullable float" The value of that is that you could have floating point numbers (things with decimals basically) but also have a functional value called 'null' when there is no data for an x or y.

## Method

_Here I try to describe what I am doing, in mostlyplain language: which data, what steps, what assumptions. A reader should be able to follow without reading the code but it can be hard to make it complete without being a little technical in the language._

I am going to use the package DuckDB for a first-pass at profiling what is in this data. Pandas would work, but DuckDB is a good fit for fast cross-file summaries without loading everything into memory. This is not a knock on Pandas!  It is one of the most important basic tools for tabular data.  For cross-file summaries, however, DuckDB really outperforms it in speed and memory efficiency. This is mostly because Pandas loads files entirely into RAM as row-based objects while DuckDB reads only required columns from disk using _"vectorized execution"_ (which means it operates on a bunch of data at once rather than one-by-one).

I'll start by making a DuckDB connection to the data:

In [5]:
con = duckdb.connect()

csv_glob = str(DATA_DIR / "pbp*.csv")

con.execute(f"""
CREATE OR REPLACE VIEW pbp_raw AS
SELECT
    *,
    regexp_extract(filename, 'pbp([0-9]{{4}})\\.csv', 1)::INTEGER AS season_from_file,
    filename AS source_file
FROM read_csv_auto(
    '{csv_glob}',
    filename = true,
    union_by_name = true,
    all_varchar = true
)
""")

Next I'll make some useful tables:

In [6]:
season_counts = con.sql("""
SELECT
    season_from_file AS season,
    COUNT(*) AS n_events,
    COUNT(DISTINCT gameid) AS n_games,
    COUNT(DISTINCT playerid) AS n_player_ids,
    COUNT(DISTINCT team) AS n_teams_or_team_codes
FROM pbp_raw
GROUP BY season_from_file
ORDER BY season_from_file
""").df()

display(season_counts)

,season,n_events,n_games,n_player_ids,n_teams_or_team_codes
0,1997,595362,1259,521,29
1,1998,605605,1260,505,29
2,1999,378643,790,507,29
3,2000,615138,1264,514,29
4,2001,602342,1260,519,29
5,2002,595588,1260,521,29
6,2003,606687,1277,502,29
7,2004,600586,1270,513,29
8,2005,630209,1314,537,30
9,2006,625271,1319,529,30


In [7]:
event_mix = con.sql("""
SELECT
    season_from_file AS season,
    lower(type) AS event_type,
    COUNT(*) AS n
FROM pbp_raw
GROUP BY 1, 2
ORDER BY season, n DESC
""").df()

display(event_mix.head(50))

,season,event_type,n
0,1997,rebound,125452
1,1997,missed shot,108776
2,1997,made shot,90474
3,1997,free throw,64005
4,1997,foul,57683
5,1997,substitution,44783
6,1997,turnover,39219
7,1997,NaN,32854
8,1997,timeout,16076
9,1997,period,10102


In [8]:
sample = con.sql("""
SELECT *
FROM pbp_raw
USING SAMPLE 20_000 ROWS
""").df()

display(sample.head())
display(sample.dtypes)
display(sample.isna().mean().sort_values(ascending=False).to_frame("missing_fraction"))

,gameid,period,clock,h_pts,a_pts,team,playerid,player,type,subtype,result,x,y,dist,desc,season,filename,season_from_file,source_file
0,0029901030,2,PT03M11.00S,0.0,0.0,GSW,679,J. Caffey,Rebound,Unknown,NaN,0,0,0,Caffey REBOUND (Off:2 Def:4),2000,../data_archive/pbp2000.csv,2000,../data_archive/pbp2000.csv
1,0022200915,1,PT03M25.00S,NaN,NaN,NaN,1610612760,NaN,Rebound,Unknown,NaN,0,0,0,THUNDER Rebound,2023,../data_archive/pbp2023.csv,2023,../data_archive/pbp2023.csv
2,0020700489,2,PT10M10.00S,0.0,0.0,UTA,2078,J. Hart,Missed Shot,Running Hook Shot,Missed,44,26,5,MISS Hart 5' Running Hook Shot,2008,../data_archive/pbp2008.csv,2008,../data_archive/pbp2008.csv
3,0020700570,1,PT03M43.00S,NaN,NaN,CLE,2544,L. James,Rebound,Unknown,NaN,0,0,0,James REBOUND (Off:1 Def:1),2008,../data_archive/pbp2008.csv,2008,../data_archive/pbp2008.csv
4,0022200027,3,PT09M17.00S,NaN,NaN,POR,1628404,J. Hart,Rebound,Unknown,NaN,0,0,0,Hart REBOUND (Off:2 Def:4),2023,../data_archive/pbp2023.csv,2023,../data_archive/pbp2023.csv


gameid                str
period                str
clock                 str
h_pts                 str
a_pts                 str
team                  str
playerid              str
player                str
type                  str
subtype               str
result                str
x                     str
y                     str
dist                  str
desc                  str
season                str
filename              str
season_from_file    int32
source_file           str
dtype: object

,missing_fraction
result,0.65285
h_pts,0.33935
a_pts,0.33935
subtype,0.14135
team,0.08590
player,0.08495
type,0.05280
season,0.03625
desc,0.00005
x,0.00005


#### Stuff that might be interesting to show the pod to start discussions of what to look into:

| Table           | Purpose                                            |
| --------------- | -------------------------------------------------- |
| `inventory`     | Which files are present and how large they are     |
| `schema_df`     | Croissant-derived column documentation             |
| `season_counts` | Events, games, players, teams by season            |
| `event_mix`     | Counts of `type` / `subtype` by season             |
| `missingness`   | Missingness by column, especially `x`, `y`, `dist` |
| `sample_game`   | One full game sorted by period and clock           |
| `final_scores`  | One row per game with final `h_pts`, `a_pts`       |
| `shot_sample`   | Rows with coordinates and shot-like event types    |


## Create a _Cleaning Layer_

I would not overwrite the raw data. Create a place to put the cleaned/wrangled data partitioned in a way that makes sense.  In this case, I would keep the organization by season unless a problem had been defined that was really not conducive to thinking that way.   cleaned Parquet layer, partitioned by season.

Here are some cleaning actions that seem likely to be useful:

1. Read IDs as strings. gameid and playerid should not be treated as numeric identifiers.
2. Normalize column names to lowercase.
3. Preserve source_file and row order.
4. Convert score, period, season, coordinates, and distance to numeric.
5. Parse clock into seconds remaining in period.
6. Derive game time elapsed.
7. Derive score margin.
8. Normalize text fields with stripped whitespace and consistent casing.
9. Treat missing x, y, and dist as expected for non-shot events, not automatically as bad data.
10. Flag suspicious games where scores decrease, periods are odd, or clocks cannot be parsed.

## Observation

_What you saw. Tables and plots are evidence — every plot gets labelled axes, units, and a title._

## Conclusion

_What you concluded, and what you're still unsure about. The "unsure about" part feeds your PR description._

_Before committing: Kernel → Restart Kernel and Run All Cells. If it doesn't run clean top-to-bottom, it isn't done._